# Lab 11 — MLlib: tuning, persistence and streaming scoring

## 1. Setup

This notebook runs on its own: the cells below recreate the required state from the first part.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder
    .appName("Lab 11 - MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:48:17 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:48:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:48:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/04 18:48:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.0.1


In [2]:
N = 200_000
REGIONS = ["North", "Northeast", "Midwest", "Southeast", "South"]

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

base = (
    spark.range(N).withColumnRenamed("id", "customer_id")
    .withColumn("tenure_months",   (F.rand(1) * 71 + 1).cast("int"))
    .withColumn("plan",            pick(["Basic", "Standard", "Premium"], 2))
    .withColumn("region",          pick(REGIONS, 3))
    .withColumn("monthly_spend",   F.round(
        F.when(F.col("plan") == "Basic", 40).when(F.col("plan") == "Standard", 120).otherwise(300)
        * (0.5 + F.rand(4)), 2))
    .withColumn("n_orders_90d",    (F.rand(5) * 15).cast("int"))
    .withColumn("support_tickets", (F.pow(F.rand(6), 2) * 9).cast("int"))
    .withColumn("has_discount",    F.rand(7) < 0.3)
    .withColumn("last_rating",     (F.rand(8) * 5 + 1).cast("int"))
)

logit = (
    0.8
    - 0.045 * F.col("tenure_months")
    + 0.45  * F.col("support_tickets")
    - 0.10  * F.col("n_orders_90d")
    - 0.45  * (F.col("last_rating") - 3)
    + F.when(F.col("plan") == "Basic", 0.5).when(F.col("plan") == "Premium", -0.6).otherwise(0.0)
    - 0.5   * F.col("has_discount").cast("int")
    + 0.6   * F.randn(9)
)

customers = (
    base
    .withColumn("churn", (F.rand(10) < 1 / (1 + F.exp(-logit))).cast("int"))
    .withColumn("last_rating",   F.when(F.rand(11) < 0.06, None).otherwise(F.col("last_rating")))
    .withColumn("monthly_spend", F.when(F.rand(12) < 0.03, None).otherwise(F.col("monthly_spend")))
)

customers.write.mode("overwrite").parquet("/tmp/lab10/customers")
customers = spark.read.parquet("/tmp/lab10/customers")
customers.show(5)

+-----------+-------------+--------+---------+-------------+------------+---------------+------------+-----------+-----+
|customer_id|tenure_months|    plan|   region|monthly_spend|n_orders_90d|support_tickets|has_discount|last_rating|churn|
+-----------+-------------+--------+---------+-------------+------------+---------------+------------+-----------+-----+
|      33333|            2|   Basic|Northeast|        39.35|          10|              0|        true|          2|    1|
|      33334|            7| Premium|  Midwest|       370.26|           6|              5|        true|          5|    1|
|      33335|           55|Standard|Southeast|         90.4|           0|              3|       false|          4|    0|
|      33336|           39| Premium|Southeast|       279.38|           3|              8|       false|          1|    1|
|      33337|           24| Premium|  Midwest|       375.69|          14|              0|        true|          1|    0|
+-----------+-------------+-----

In [3]:
train, test = customers.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train: {train.count():,} | Test: {test.count():,}")

Train: 160,091 | Test: 39,909


In [4]:
sample = train.limit(5)

imputer = Imputer(strategy="median",
                  inputCols=["last_rating", "monthly_spend"],
                  outputCols=["last_rating_imp", "monthly_spend_imp"])
step1 = imputer.fit(train).transform(sample)
step1.select("last_rating", "last_rating_imp", "monthly_spend", "monthly_spend_imp").show()

+-----------+---------------+-------------+-----------------+
|last_rating|last_rating_imp|monthly_spend|monthly_spend_imp|
+-----------+---------------+-------------+-----------------+
|          2|              2|        39.35|            39.35|
|          5|              5|       370.26|           370.26|
|          1|              1|       279.38|           279.38|
|          1|              1|       375.69|           375.69|
|          3|              3|       109.48|           109.48|
+-----------+---------------+-------------+-----------------+



In [5]:
indexer = StringIndexer(inputCols=["plan", "region"], outputCols=["plan_idx", "region_idx"],
                        handleInvalid="keep")          # new category in the future -> extra index
encoder = OneHotEncoder(inputCols=["plan_idx", "region_idx"], outputCols=["plan_vec", "region_vec"])

idx_model = indexer.fit(train)
print("Category order (from most to least frequent):", idx_model.labelsArray)

step2 = encoder.fit(idx_model.transform(train)).transform(idx_model.transform(sample))
step2.select("plan", "plan_idx", "plan_vec", "region", "region_vec").show(truncate=False)

Category order (from most to least frequent): [['Premium', 'Standard', 'Basic'], ['South', 'Southeast', 'Midwest', 'Northeast', 'North']]
+--------+--------+-------------+---------+-------------+
|plan    |plan_idx|plan_vec     |region   |region_vec   |
+--------+--------+-------------+---------+-------------+
|Basic   |2.0     |(3,[2],[1.0])|Northeast|(5,[3],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Midwest  |(5,[2],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Southeast|(5,[1],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Midwest  |(5,[2],[1.0])|
|Standard|1.0     |(3,[1],[1.0])|South    |(5,[0],[1.0])|
+--------+--------+-------------+---------+-------------+



In [6]:
NUMERIC = ["tenure_months", "monthly_spend_imp", "n_orders_90d", "support_tickets", "last_rating_imp"]

assembler = VectorAssembler(inputCols=NUMERIC + ["has_discount", "plan_vec", "region_vec"],
                            outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False)

prep = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler]).fit(train)
prep.transform(sample).select("customer_id", "features").show(3, truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id|features                                                                                                                                                                     |
+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|33333      |(14,[0,1,2,4,5,8,12],[0.09761743173080811,0.32815511501943667,2.3129241591698517,1.4570098078641665,2.1822279142728385,2.1221270315598786,2.502375151166683])                |
|33334      |(14,[0,1,2,3,4,5,6,11],[0.3416610110578284,3.0877436565971186,1.387754495501911,1.908562301064785,3.642524519660416,2.1822279142728385,2.1203092930769682,2.498127448835056])|
|33336      |(14,[0,1,2,3,4,6,10],[1.9035399187507582,2.3298

In [7]:
lr = LogisticRegression(featuresCol="features", labelCol="churn", maxIter=50)
lr_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, lr])

t0 = time.perf_counter()
lr_model = lr_pipeline.fit(train)
print(f"Training: {time.perf_counter() - t0:.1f} s")

pred = lr_model.transform(test)
pred.select("customer_id", "churn", "probability", "prediction").show(5, truncate=False)

Training: 2.2 s
+-----------+-----+----------------------------------------+----------+
|customer_id|churn|probability                             |prediction|
+-----------+-----+----------------------------------------+----------+
|33335      |0    |[0.671194288271684,0.32880571172831596] |0.0       |
|33339      |0    |[0.11116990372080929,0.8888300962791907]|1.0       |
|33341      |1    |[0.1516879824018518,0.8483120175981482] |1.0       |
|33346      |1    |[0.33894464016172376,0.6610553598382762]|1.0       |
|33352      |0    |[0.7561792884851825,0.24382071151481755]|0.0       |
+-----------+-----+----------------------------------------+----------+
only showing top 5 rows


In [8]:
def evaluate(predictions, name="model"):
    auc_roc = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC").evaluate(predictions)
    auc_pr  = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderPR").evaluate(predictions)
    mc = lambda m, **kw: MulticlassClassificationEvaluator(labelCol="churn", metricName=m, **kw).evaluate(predictions)
    print(f"{name:<22} AUC-ROC={auc_roc:.3f}  AUC-PR={auc_pr:.3f}  "
          f"accuracy={mc('accuracy'):.3f}  precision={mc('precisionByLabel', metricLabel=1.0):.3f}  "
          f"recall={mc('recallByLabel', metricLabel=1.0):.3f}")

evaluate(pred, "Logistic regression")

print("\nConfusion matrix:")
pred.groupBy("churn").pivot("prediction", [0.0, 1.0]).count().orderBy("churn").show()

Logistic regression    AUC-ROC=0.831  AUC-PR=0.778  accuracy=0.760  precision=0.731  recall=0.655

Confusion matrix:
+-----+-----+-----+
|churn|  0.0|  1.0|
+-----+-----+-----+
|    0|19641| 3932|
|    1| 5636|10700|
+-----+-----+-----+



---
## 2. Hyperparameter tuning with cross-validation

```
  training set split into k = 3 parts (folds)

  parameter combination 1:     [train  ][train  ][valid  ]  -> AUC_1
                               [train  ][valid  ][train  ]  -> AUC_2
                               [valid  ][train  ][train  ]  -> AUC_3   => mean

  parameter combination 2:     ...                                    => mean

  the best mean is retrained on the whole training set; the TEST set is only used at the end
```

Cost: `combinations x folds` trainings. The `parallelism` parameter trains several combinations at the same time.

In [9]:
grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.0, 0.01, 0.1])
        .addGrid(lr.elasticNetParam, [0.0, 1.0])     # 0 = ridge (L2), 1 = lasso (L1)
        .build())

cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=grid,
    evaluator=BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC"),
    numFolds=3,
    parallelism=2,
    seed=42,
)

t0 = time.perf_counter()
cv_model = cv.fit(train)
print(f"{len(grid)} combinations x 3 folds: {time.perf_counter() - t0:.1f} s\n")

for params, metric in sorted(zip(grid, cv_model.avgMetrics), key=lambda x: -x[1]):
    print(f"Mean AUC = {metric:.4f} | " + ", ".join(f"{p.name}={v}" for p, v in params.items()))

best = cv_model.bestModel
evaluate(best.transform(test), "\nBest LR (test)")

6 combinations x 3 folds: 14.7 s

Mean AUC = 0.8337 | regParam=0.0, elasticNetParam=1.0
Mean AUC = 0.8337 | regParam=0.0, elasticNetParam=0.0
Mean AUC = 0.8337 | regParam=0.01, elasticNetParam=0.0
Mean AUC = 0.8332 | regParam=0.01, elasticNetParam=1.0
Mean AUC = 0.8331 | regParam=0.1, elasticNetParam=0.0
Mean AUC = 0.7786 | regParam=0.1, elasticNetParam=1.0

Best LR (test)        AUC-ROC=0.831  AUC-PR=0.778  accuracy=0.760  precision=0.731  recall=0.655


Regularization did not help: with 160 thousand rows and 14 features, there is no overfitting to fight. It makes a difference with many features, little data or correlated features. The process, however, is the same for any model and parameter grid.

---
## 3. Saving, loading and using the model

The saved `PipelineModel` contains **all the stages**: Imputer medians, StringIndexer categories, scales and coefficients. Whoever loads the model applies exactly the same preparation, without rewriting code.

In [10]:
MODEL_PATH = "/tmp/lab10/models/churn_lr"
best.write().overwrite().save(MODEL_PATH)

loaded = PipelineModel.load(MODEL_PATH)
print("Stages:", [type(s).__name__ for s in loaded.stages])

# New customers, including a null and a region that did not exist in training
new_customers = spark.createDataFrame(
    [(900001, 3,  "Basic",   "South",   None,  1, 6, False, 1),
     (900002, 60, "Premium", "North",   410.0, 12, 0, True, 5),
     (900003, 12, "Standard","Islands", 95.0,  4, 2, False, None)],
    "customer_id long, tenure_months int, plan string, region string, monthly_spend double, "
    "n_orders_90d int, support_tickets int, has_discount boolean, last_rating int",
)

(loaded.transform(new_customers)
 .select("customer_id", "plan", "support_tickets",
         F.round(vector_to_array("probability")[1], 3).alias("p_churn"))
 .show())

Stages: ['ImputerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel', 'LogisticRegressionModel']
+-----------+--------+---------------+-------+
|customer_id|    plan|support_tickets|p_churn|
+-----------+--------+---------------+-------+
|     900001|   Basic|              6|  0.987|
|     900002| Premium|              0|  0.008|
|     900003|Standard|              2|  0.673|
+-----------+--------+---------------+-------+



### 3.1 Streaming scoring

A `PipelineModel` is a Transformer, and Transformers work on streaming DataFrames. Connecting with Lab 9: each customer arriving in a folder gets a risk score within seconds.

In [11]:
import os, shutil
STREAM_IN = "/tmp/lab10/stream_in"
STAGING   = "/tmp/lab10/staging"
for d in [STREAM_IN, STAGING]:
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)

stream = spark.readStream.schema(new_customers.schema).json(STREAM_IN)

q = (loaded.transform(stream)
     .select("customer_id", F.round(vector_to_array("probability")[1], 3).alias("p_churn"))
     .filter("p_churn >= 0.5")                                # only at-risk customers
     .writeStream.format("memory").queryName("at_risk").outputMode("append").start())

# atomic write (Lab 9): writes outside the watched folder and then moves it
with open(f"{STAGING}/batch1.json", "w") as f:
    for row in new_customers.toJSON().collect():
        f.write(row + "\n")
os.replace(f"{STAGING}/batch1.json", f"{STREAM_IN}/batch1.json")

q.processAllAvailable()
spark.sql("SELECT * FROM at_risk").show()
q.stop()

+-----------+-------+
|customer_id|p_churn|
+-----------+-------+
|     900001|  0.987|
|     900003|  0.673|
+-----------+-------+



---
## Summary

| Step | MLlib |
|-------|-------|
| Split | `df.randomSplit([0.8, 0.2], seed=42)` |
| Nulls | `Imputer(strategy="median")` |
| Categories | `StringIndexer(handleInvalid="keep")` + `OneHotEncoder` |
| Feature vector | `VectorAssembler` |
| Scale | `StandardScaler` |
| Derived features in the pipeline | `SQLTransformer` |
| Classification | `LogisticRegression`, `RandomForestClassifier`, `GBTClassifier` |
| Evaluation | `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` |
| Tuning | `ParamGridBuilder` + `CrossValidator` |
| Probability as a number | `vector_to_array("probability")[1]` |
| Persistence | `model.write().overwrite().save(path)` / `PipelineModel.load(path)` |

In [13]:
spark.stop()